# 국민연금 2027 목표비중 — 통합 재현 노트북

이 노트북은 GitHub 저장소의 버전관리된 `src/` 계산 스크립트를 순서대로 실행하고, 각 단계의 핵심 결과를 표로 확인한다.

실행 순서: **STEP 2 → STEP 5~8 → STEP 9 → STEP 10 → STEP 11 → STEP 12 → STEP 13 → STEP 14 제출파일 생성**

> 팀 분석 가정: EM≤10%, PE≤10%, 인프라≤10%, PD≤10%. 이는 국민연금 공식 세부한도가 아니다.  
> STEP 5~10은 현금 0.1%를 제외한 위험자산 슬리브를 정규화하여 비교하고, STEP 11 이후 현금 0.1%를 복원한다.



In [ ]:
from pathlib import Path
import runpy, json
import pandas as pd
from IPython.display import display

ROOT=Path.cwd()
RESULTS=ROOT/"results"
SUBMISSION=ROOT/"submission"
RESULTS.mkdir(exist_ok=True)
SUBMISSION.mkdir(exist_ok=True)

def show_csv(path, n=20):
    p=ROOT/path
    print("\n",path)
    df=pd.read_csv(p)
    display(df.head(n))
    return df

print("root:",ROOT)


## STEP 2 — 프록시 수익률·상관·CMA 공분산

데이터가 없으면 yfinance에서 2016-09~2026-08의 120개월 프록시 데이터를 다시 수집한다. 해외자산은 원화 비헤지 기준으로 환산한다.



In [ ]:
runpy.run_path(str(ROOT/"src/step2_build_corr.py"), run_name="__main__")
show_csv("data/diagnostics.csv")


## STEP 5~8 — 기준 MVO, Ledoit–Wolf, Box, Ellipsoid


In [ ]:
runpy.run_path(str(ROOT/"src/analyze_steps5_8.py"), run_name="__main__")
show_csv("results/step5_mvo.csv")
show_csv("results/step6_lw_diagnostics.csv")
show_csv("results/step7_box.csv")
show_csv("results/step8_ellipsoid.csv")


## STEP 9 — Michaud 300회 재표본 + 반복별 Ledoit–Wolf


In [ ]:
runpy.run_path(str(ROOT/"src/step9_michaud.py"), run_name="__main__")
show_csv("results/step9_michaud_summary.csv")


## STEP 10 — 방법론 종합비교


In [ ]:
runpy.run_path(str(ROOT/"src/step10_method_synthesis.py"), run_name="__main__")
show_csv("results/step10_tableB_weights.csv")
show_csv("results/step10_tableB_metrics.csv")
show_csv("results/step10_mu_50bp_sensitivity_summary.csv")
show_csv("results/step10_small_eigenvectors.csv")


## STEP 11 — 2027 팀 목표비중 시나리오

팀 목표는 어느 한 최적화 결과를 그대로 복사하지 않고 규모·시장지분·대체투자 집행·환위험·이행가능성을 추가 반영한다.



In [ ]:
runpy.run_path(str(ROOT/"src/step11_team_target.py"), run_name="__main__")
show_csv("results/step11_tableC_common.csv")
show_csv("results/step11_tableC_detailed.csv")


## STEP 12 — Policy Black–Litterman

prior는 STEP 11 팀 목표가 아니라 **2027 official mapped**를 사용한다.  
기준: δ=2.5, τ=0.025, P=I, T=10년, Ω=Σ/T, 자산별 active ±3%p, TE≤1.0%.



In [ ]:
runpy.run_path(str(ROOT/"src/step12_policy_bl.py"), run_name="__main__")
show_csv("results/step12_tableD.csv")
show_csv("results/step12_allocations.csv")
show_csv("results/step12_metrics.csv")


## STEP 13 — 스트레스 테스트

1. CMA가 Robust BL active 방향에 불리하게 1SE 빗나감  
2. 주식 기대수익률 -2%p, 주식 상관 +0.15  
3. 대체 변동성 +50%, 대체-글로벌주식 상관 +0.20



In [ ]:
runpy.run_path(str(ROOT/"src/step13_stress_test.py"), run_name="__main__")
show_csv("results/step13_stress_results.csv")
show_csv("results/step13_stress1_active_loss.csv")
show_csv("results/step13_covariance_diagnostics.csv")


## STEP 14 — 최종 의결 및 제출 CSV

최종 분석 시나리오는 Robust BL 비중을 사용한다.

재심의 조건:
1. 공식 목표 대비 사전 TE > 1.0%
2. 최근 8분기 뷰 적중률 < 55%
3. 국내주식 목표 이행에 필요한 시장참여율 > 일평균 거래대금의 1%



In [ ]:
common=pd.read_csv(RESULTS/"step12_common_buckets.csv",index_col=0)
tableD=pd.read_csv(RESULTS/"step12_tableD.csv")

official={"국내주식":0.208,"해외주식":0.356,"국내채권":0.218,
          "해외채권":0.074,"대체투자":0.143,"단기자금":0.001}
team=pd.read_csv(RESULTS/"step11_tableC_common.csv",index_col=0)["Team scenario"].to_dict()
final=common["Robust BL"].to_dict()

weights=pd.DataFrame({
    "asset":list(official.keys()),
    "official_2027":[official[k] for k in official],
    "team_step11":[team[k] for k in official],
    "final_decision":[final[k] for k in official],
})
weights.to_csv(SUBMISSION/"team2_weights.csv",index=False,encoding="utf-8-sig")

views=tableD[["asset","CMA_total","Q_excess","pi_excess",
              "view_error_Q_minus_pi","mu_BL_total","Omega_diag","V_BL_diag"]].copy()
views.to_csv(SUBMISSION/"team2_views.csv",index=False,encoding="utf-8-sig")

resolution=(
"위원회는 2027년 공식 목표비중을 일부 수정하여 국내주식 19.2%, 해외주식 32.8%, "
"국내채권 24.8%, 해외채권 9.3%, 대체투자 13.8%, 단기자금 0.1%로 설정한다. "
"이는 팀 CMA, MVO 추정오차 검증, Ledoit–Wolf·Robust·Michaud, 공식 목표를 사전비중으로 둔 "
"Policy Black–Litterman 및 세 가지 스트레스 결과를 종합한 조건부 의결이다. "
"다만 공식 목표 대비 사전 추적오차가 1.0%를 초과하거나 최근 8분기 뷰 적중률이 55% 미만이거나 "
"국내주식 목표 이행에 필요한 시장참여율이 일평균 거래대금의 1%를 초과하면 공식 목표로 복귀하거나 재심의한다."
)

summary={
    "final_common_weights":final,
    "review_conditions":{
        "tracking_error_max":0.01,
        "view_hit_rate_window_quarters":8,
        "view_hit_rate_min":0.55,
        "domestic_equity_market_participation_max":0.01
    },
    "resolution":resolution
}
(RESULTS/"step14_final_resolution.json").write_text(
    json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8"
)

print(resolution)
display(weights)
display(views)
print("saved:",SUBMISSION/"team2_weights.csv",SUBMISSION/"team2_views.csv")
